# Project: Calculator Agent

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will build a complete calculator agent with math tools, a tool-calling loop, and `MessagesState` to solve arithmetic problems.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define Math Tools

Three tools for basic arithmetic. The LLM reads the docstrings to decide which tool to call.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together. Use this for addition operations."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together. Use this for multiplication operations."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Use this for division operations."""
    return a / b

tools = [add, multiply, divide]

## 4. Create the LLM with Bound Tools

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

## 5. Define the Calculator Node

In [ ]:
from langgraph.graph import MessagesState

def calculator_node(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

## 6. Build and Compile the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

graph = StateGraph(MessagesState)

graph.add_node("calculator", calculator_node)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "calculator")
graph.add_conditional_edges("calculator", tools_condition)
graph.add_edge("tools", "calculator")

app = graph.compile()

## 7. Visualize the Graph

In [ ]:
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

## 8. Run Calculations

Test the agent with several math problems.

In [ ]:
from langchain_core.messages import HumanMessage

questions = [
    "What is 42 * 17?",
    "What is 1000 / 8?",
    "What is 25 + 75 * 2?",
    "What is (144 / 12) + (8 * 7)?",
]

for question in questions:
    result = app.invoke({"messages": [HumanMessage(content=question)]})
    answer = result["messages"][-1].content
    print(f"Q: {question}")
    print(f"A: {answer}\n")

## 9. Trace a Single Run

Print every message to see the full agent loop in action.

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="What is (100 / 4) + (15 * 3)?")]
})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

## What You Built

1. Three **`@tool`-decorated** math functions (add, multiply, divide)
2. An LLM node with **`bind_tools`** for structured tool calling
3. A **`ToolNode`** that executes tool calls and returns results
4. A **conditional edge** via `tools_condition` that creates the agent loop
5. A complete calculator agent that solves multi-step arithmetic problems